# 11a · The settings test (shared setup)

The 52 hidden-match tests used to choose every setting: 2 forward-in-time tests (learn 2022–23, test 2024; learn
2022–24, test 2025) and 50 random half-and-half splits of matches. Also the yardstick: how players did on pressure
balls in the matches they were hidden from. Notebooks 11 and 15 both run this first.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repository root, where the small rrlib helper package lives
import rrlib.common

<details><summary>Technical notes</summary>

P9 - Experiment grid with a pre-registered, staged selection protocol (Stage 2 of the plan).  
Pressure is already locked (P4). Every remaining choice is judged on ONE yardstick: the player's de-drifted  
win-probability added above expected (V-2) on pressure balls in matches the method never saw  
(run value above expected, V-1, is the second check).  
Splits: rolling-origin (2022-23 -> 2024, 2022-24 -> 2025) + 50 random 50/50 match splits within 2022-25.  
2026 is sealed and opened once at the end for the chosen configuration (and simple baselines).  
Stages: A value metric -> B rating model -> C gap weight w_d and consistency weight w_c -> D shrinkage and minimum  
sample -> E local sensitivity. Rule: best mean objective; a simpler option within one paired SE wins.
</details>

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import time
import numpy as np
import pandas as pd
from rrlib.common import CFG, SEED, load, save_table, save_json, load_json, get_logger, Timer, weighted_spearman
from rrlib import ratings as RL

log = get_logger("p09_experiments")
V_ALL = load("values")
comp_info = load_json("p05_components")
HOLD = CFG["cv"]["holdout_season"]
V = V_ALL[V_ALL.year < HOLD].copy()
MIN_TEST = 20

#### Splits

In [ ]:
splits = []
for tr_years, te_year in CFG["cv"]["rolling"]:
    splits.append(("rolling", f"{tr_years[0]}-{tr_years[-1]}->{te_year}", V.year.isin(tr_years).values, (V.year == te_year).values))
mids = np.sort(V.match_id.unique())
rng = np.random.RandomState(SEED)
for i in range(CFG["cv"]["random_splits"]):
    a = set(rng.choice(mids, len(mids) // 2, replace=False))
    m = V.match_id.isin(a).values
    splits.append(("random", f"r{i}", m, ~m))
log.info(f"{len(splits)} splits")

In [ ]:
def yardstick(test, side):
    pcol = RL.PCOL[side]
    t = test[test.is_pressure & test[f"v2_{side}"].notna()]
    g = t.groupby(pcol).agg(y2=(f"v2_{side}", "mean"), y1=(f"v1_{side}", "mean"), n=(f"v2_{side}", "size"))
    return g[g.n >= MIN_TEST]


YARD, BASE_IDS = {}, {}
for kind, name, tr, te in splits:
    for side in ("bat", "bowl"):
        YARD[(name, side)] = yardstick(V[te], side)
        # common evaluation pool for EVERY configuration: players with a test yardstick and >=1 training pressure
        # ball. Players a configuration excludes (minimum-sample rules) are ranked last instead of being dropped,
        # so correlations are always computed on the same players.
        d = V[tr]
        BASE_IDS[(name, side)] = set(d.loc[d.is_pressure & d[f"v1_{side}"].notna(), RL.PCOL[side]].unique())

R1_CACHE = {}
_orig_r1 = RL.r1_ratings
def cached_r1(v, side, metric="v1", k_mult=1.0, lam=None):
    key = (CUR_SPLIT, metric, k_mult)
    if key not in R1_CACHE:
        R1_CACHE[key] = RL.r1_fit(v, metric, k_mult, lam)
    return R1_CACHE[key][side]
RL.r1_ratings = cached_r1
CUR_SPLIT = None

In [ ]:
def evaluate(cfg, sides=("bat", "bowl")):
    global CUR_SPLIT
    rows = []
    for kind, name, tr, te in splits:
        CUR_SPLIT = name
        train = V[tr]
        for side in sides:
            sc = RL.build_scores(train, side, cfg, comp_info).score
            y = YARD[(name, side)]
            pool = y.index[y.index.isin(BASE_IDS[(name, side)])]
            if len(pool) < 15:
                continue
            s, yy = sc.reindex(pool).fillna(-1e9), y.loc[pool]
            top = s.sort_values(ascending=False).index[:10]
            wmean = lambda col, idx: np.average(yy.loc[idx, col], weights=yy.loc[idx, "n"])
            rows.append({"kind": kind, "split": name, "side": side, "n_pool": len(pool),
                         "rho_v2": weighted_spearman(s.values, yy.y2.values, yy.n.values),
                         "rho_v1": weighted_spearman(s.values, yy.y1.values, yy.n.values),
                         "top10_lift_v2": wmean("y2", top) - wmean("y2", pool),
                         "top10_lift_v1": wmean("y1", top) - wmean("y1", pool)})
    return pd.DataFrame(rows)

In [ ]:
def objective(res):
    """per side: 0.5 * mean over random splits + 0.5 * mean over rolling splits of rho_v2; averaged over sides."""
    o = {}
    for side in res.side.unique():
        r = res[res.side == side]
        o[side] = 0.5 * r[r.kind == "random"].rho_v2.mean() + 0.5 * r[r.kind == "rolling"].rho_v2.mean()
    return o


SIMPLE = {"value": {"v1": 1, "v2": 1, "v4": 2, "v3": 3},
          "model": {"R2-p_only": 1, "R2-joint": 2, "R2-weighted": 3, "R3": 3, "R1": 4, "blend": 5}}

In [ ]:
def complexity(cfg):
    return (SIMPLE["value"].get(cfg.get("value"), 1) + SIMPLE["model"].get(cfg.get("model"), 1)
            + (cfg.get("w_d", 0) > 0) + (cfg.get("w_c", 0) > 0) + (cfg.get("k_mult", 1) != 1) + (cfg.get("min_n", 0) > 0)
            + 0.1 * abs(np.log2(cfg.get("alpha", 1.0))))

In [ ]:
def run_stage(label, cfgs, sides=("bat", "bowl")):
    out, per = [], {}
    for cfg in cfgs:
        t0 = time.time()
        res = evaluate(cfg, sides)
        per[str(cfg)] = res
        for sd in sides:
            r = res[res.side == sd]
            out.append({"stage": label, "side": sd, "config": str(cfg), **{k: cfg.get(k) for k in ("value", "model", "w_d", "w_c", "k_mult", "min_n", "alpha")},
                        "objective": objective(res)[sd],
                        "rho_v2_random": r[r.kind == "random"].rho_v2.mean(), "rho_v2_rolling": r[r.kind == "rolling"].rho_v2.mean(),
                        "rho_v1_random": r[r.kind == "random"].rho_v1.mean(), "rho_v1_rolling": r[r.kind == "rolling"].rho_v1.mean(),
                        "top10_lift_v2_x100": 100 * r.top10_lift_v2.mean(), "top10_lift_v1": r.top10_lift_v1.mean(),
                        "complexity": complexity(cfg)})
        log.info(f"[{label}] {cfg} -> " + ", ".join(f"{o['side']} {o['objective']:.4f}" for o in out[-len(sides):]) + f" ({time.time() - t0:.0f}s)")
    return pd.DataFrame(out), per

In [ ]:
def choose(df, per, side):
    d = df[df.side == side].sort_values("objective", ascending=False).reset_index(drop=True)
    best = d.iloc[0]
    rb = per[best.config]
    rb = rb[(rb.side == side) & (rb.kind == "random")].set_index("split").rho_v2
    pick = best
    for _, r in d.iterrows():
        if r.complexity >= pick.complexity:
            continue
        rr = per[r.config]
        rr = rr[(rr.side == side) & (rr.kind == "random")].set_index("split").rho_v2
        diff = (rb - rr.reindex(rb.index))
        se = diff.std() / np.sqrt(diff.notna().sum())
        if best.objective - r.objective <= se:
            pick = r
    return eval(pick.config), d